In [41]:
raw_prescriptions = [
    "patient_id=101; drug=Aspirin; dose=100; unit=mg; date=2024-01-10",
    "patient_id=102; drug=Paracetamol; dose=500; unit=mg; date=2024-01-11",
    "patient_id=101; drug=Aspirin; dose=100; unit=mg; date=2024-01-12",
    "patient_id=103; drug=Aspirin; dose=invalid; unit=mg; date=2024-01-13",
    "patient_id=101; drug=Ibuprofen; dose=200; unit=mg; date=2024-01-14",
    "MALFORMED LINE",
    "patient_id=102; drug=Paracetamol; dose=500; unit=mg; date=2024-01-15",
    "patient_id=101; drug=Aspirin; dose= ; unit=mg; date=2024-01-16",
]

In [112]:
def transform_data(raw_data: list[str]) -> dict:
    out = {}

    for line in raw_data:
        try:
            parts = line.split(";")
            data_dict = {}

            for part in parts:
                key, value = part.split("=")
                data_dict[key.strip()] = value.strip()

            patient_id = data_dict["patient_id"]
            drug = data_dict["drug"]
            dose = int(data_dict["dose"])

            if patient_id not in out:
                out[patient_id] = {}

            if drug not in out[patient_id]:
                out[patient_id][drug] = {"count": 0, "total_dose": 0}

            out[patient_id][drug]["count"] += 1
            out[patient_id][drug]["total_dose"] += dose

        except Exception:
            continue

    return out

In [123]:
for patient_id, drugs in trans_data.items():
    print(f"\nPatient : {patient_id}")
    
    for drug, stats in drugs.items():
        print(f"  Médicament : {drug}")
        print(f"    Nombre total de prescriptions : {stats['count']}")
        print(f"    Dose totale cumulée : {stats['total_dose']}")


Patient : 101
  Médicament : Aspirin
    Nombre total de prescriptions : 2
    Dose totale cumulée : 200
  Médicament : Ibuprofen
    Nombre total de prescriptions : 1
    Dose totale cumulée : 200

Patient : 102
  Médicament : Paracetamol
    Nombre total de prescriptions : 2
    Dose totale cumulée : 1000


In [125]:
print("\nPatients ayant reçu plus de 2 prescriptions d'Aspirin :")

for patient_id, drugs in trans_data.items():
    if "Aspirin" in drugs and drugs["Aspirin"]["count"]>= 2:
        print(f"Patient : {patient_id}")


Patients ayant reçu plus de 2 prescriptions d'Aspirin :
Patient : 101


In [142]:
import re
import unicodedata

def clean_string(text: str) -> str:
    # lowercase + normalisation des espaces
    txt = text.lower()
    txt = re.sub(r'\s+', " ", txt)
    return txt.strip()


def handle_punctuation(text: str) -> str:
    # on enlève uniquement la ponctuation simple
    # sans supprimer d'information linguistique comme "d'"
    txt = re.sub(r"[^\w\s']", "", text)
    return txt

def remove_accents(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )


def detect_patho(text: str, patho: str, negation: list[str]) -> bool:
    if patho not in text:
        return False

    # Vérifier si la pathologie est précédée par une négation
    for neg in negation:
        # On cherche la négation dans une fenêtre avant la pathologie
        pattern = rf"{neg}[^\.]*{patho}"
        if re.search(pattern, text):
            return False

    return True


def detect(reports: list[str],
           target_conditions: list[str],
           negations: list[str]) -> dict:

    out = {target: 0 for target in target_conditions}

    for rep in reports:
        # nettoyage
        text = clean_string(rep)
        text = handle_punctuation(text)
        text = remove_accents(text)

        # détection
        for patho in target_conditions:
            patho = remove_accents(patho)
            if detect_patho(text, patho, negations):
                out[patho] += 1

    return out

In [143]:
reports = [
    "Patient admis pour insuffisance cardiaque chronique. Pas de signe d'infection pulmonaire.",
    "Antécédents de diabète de type 2. Suspicion d'insuffisance rénale.",
    "Aucun signe de diabète. Infection pulmonaire confirmée.",
    "Insuffisance cardiaque sévère et diabète mal contrôlé.",
    "Absence d'insuffisance rénale. Par contre infection pulmonaire probable."
]

target_conditions = [
    "insuffisance cardiaque",
    "diabete",
    "infection pulmonaire",
    "insuffisance renale"
]

negations  = ["pas de","aucun", "absence de"]
detect(reports, target_conditions, negations)

{'insuffisance cardiaque': 2,
 'diabete': 2,
 'infection pulmonaire': 1,
 'insuffisance renale': 2}